In [1]:
import json
import uuid

def vqa_rad_to_llava(sample):
    return {
        "id": str(uuid.uuid4()),
        "image": sample["image_name"],
        "conversations": [
            {
                "from": "human",
                "value": "<image>\n" + sample["question"]
            },
            {
                "from": "gpt",
                "value": sample["answer"]
            }
        ],
        # 下面这两个字段不影响训练，但对你后处理非常重要
        "answer_type": sample["answer_type"],     # OPEN / CLOSED
        "question_type": sample["question_type"]  # PLANE / MODALITY / etc.
    }


In [2]:
with open("/root/data/VQA_RAD/VQA_RAD Dataset Public.json") as f:
    raw_data = json.load(f)

llava_data = [vqa_rad_to_llava(s) for s in raw_data]

with open("/root/data/VQA_RAD/vqa_rad_llava.json", "w") as f:
    json.dump(llava_data, f, indent=2)


In [6]:
import json
import random
import os

# --- 配置部分 ---
input_file = '/root/data/VQA_RAD/vqa_rad_llava.json'  # 你转换好的总文件路径
output_dir = '/root/data/VQA_RAD/dataset_split'     # 输出文件夹
split_ratio = 0.8                  # 训练集占比 (0.8 = 80% 训练, 20% 测试)
seed = 42                          # 随机种子，保证结果可复现

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

# 设置随机种子
random.seed(seed)

def split_dataset():
    # 1. 读取数据
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"total number: {len(data)}")
    
    # 2. 根据 answer_type 分类
    open_ended = []
    closed_ended = []
    
    for item in data:
        # 注意：你需要确认你的数据集中开放性问题的标记是 "OPEN" 还是其他
        # 这里的判断逻辑是：如果是 CLOSED 归为一类，其余归为 OPEN
        if item.get('answer_type') == 'CLOSED':
            closed_ended.append(item)
        else:
            open_ended.append(item)
            
    print(f"Closed number: {len(closed_ended)}")
    print(f"Open number: {len(open_ended)}")
    
    # 3. 分层切分 (Stratified Split)
    # 切分 Closed 数据
    random.shuffle(closed_ended)
    closed_split_idx = int(len(closed_ended) * split_ratio)
    train_closed = closed_ended[:closed_split_idx]
    test_closed = closed_ended[closed_split_idx:]
    
    # 切分 Open 数据
    random.shuffle(open_ended)
    open_split_idx = int(len(open_ended) * split_ratio)
    train_open = open_ended[:open_split_idx]
    test_open = open_ended[open_split_idx:]
    
    # 4. 构建最终数据集
    # 训练集：合并两者并打乱
    train_set = train_closed + train_open
    random.shuffle(train_set)
    
    # 测试集：分别保存，方便评估
    test_all = test_closed + test_open
    random.shuffle(test_all)
    
    # 5. 保存文件
    def save_json(data_list, filename):
        path = os.path.join(output_dir, filename)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data_list, f, indent=2, ensure_ascii=False)
        print(f"save: {filename} (number: {len(data_list)})")

    save_json(train_set, 'train.json')
    save_json(test_all, 'test_all.json')
    save_json(test_closed, 'test_closed.json') 
    save_json(test_open, 'test_open.json')     

if __name__ == "__main__":
    try:
        split_dataset()
        print("\nsplit over！Now we can fine-tuning")
    except FileNotFoundError:
        print(f"错误：找不到文件 {input_file}，请检查路径。")

total number: 2248
Closed number: 1297
Open number: 951
save: train.json (number: 1797)
save: test_all.json (number: 451)
save: test_closed.json (number: 260)
save: test_open.json (number: 191)

split over！Now we can fine-tuning


In [ ]:
!torchrun --nproc_per_node=1 llava/train/train.py \
  --model_name_or_path /root/autodl-tmp/llava-v1.5-7b \
  --version v1 \
  --data_path /root/data/VQA_RAD/dataset_split/train.json \
  --image_folder /root/autodl-tmp/VQA_RAD/images \
  --vision_tower /root/autodl-tmp/clip-vit-large-patch14-336 \
  --output_dir /root/output/llava-vqa-rad-lora \
  --bf16 True \
  --bits 4 \
  --per_device_train_batch_size 1 \
  --gradient_accumulation_steps 16 \
  --learning_rate 2e-4 \
  --num_train_epochs 5 \
  --lora_enable True \
  --lora_r 64 \
  --lora_alpha 128 \
  --lora_dropout 0.05 \
  --group_by_modality_length True \
  --logging_steps 1 \
  --save_steps 500 \
  --save_total_limit 2 \
  --gradient_checkpointing True \
  --dataloader_num_workers 4 \
  --model_max_length 2048